# 01 · Physics of Fringe Images & Sampling

**Sistem Fotonika — Digital Fringe Processing**  
**Iwan Cony Setiadi · Teknik Fisika**  
**Level:** S1 semester 6 dengan pengayaan riset · **Mode:** teori + implementasi + eksperimen numerik

> **Status data:** semua citra dan angka dalam notebook ini berasal dari **model/simulasi sintetis** dengan *ground truth*, bukan pengukuran instrumen nyata. Asumsi dan keterbatasan dibahas dalam setiap eksperimen.

**Tujuan:** menurunkan model interferensi, menguji visibilitas, membangkitkan fringe, dan memeriksa kesalahan kamera serta aliasing.

## Pertanyaan fisika dan capaian


**Pertanyaan:** mengapa kamera mengukur *intensitas*, tetapi mampu menyimpan informasi *fase*? Apakah semua pola bergaris memerlukan laser?

Setelah notebook ini mahasiswa dapat: (i) menurunkan suku silang interferensi dari medan kompleks; (ii) memprediksi visibilitas, termasuk akibat koherensi terbatas; (iii) mensintesis citra dengan model noise kamera; (iv) menentukan batas Nyquist dalam satuan siklus/piksel; (v) membedakan kontras fisik dari rentang tampilan digital.

**Batas model:** scalar, kuasi-monokromatik, medan dengan polarisasi searah, sensor merespons intensitas terintegrasi selama eksposur. Untuk cahaya terpolarisasi berbeda, teori vektor harus digunakan. Moiré geometris dan fringe projection tidak memerlukan interferensi koheren.

## 1 · Dari medan kompleks ke intensitas


Ambil konvensi fasor waktu $\exp(-i\omega t)$ dan medan analitik $\mathcal E_j(\mathbf r,t)=a_j(\mathbf r)\exp[i\phi_j(\mathbf r)]\exp(-i\omega t)$. Pada polarisasi sama, detektor menghasilkan intensitas rata-rata yang proporsional dengan kuadrat modulus medan:

$$I=|E_1+E_2|^2=I_1+I_2+2\sqrt{I_1I_2}\cos(\phi_1-\phi_2).$$

Jika derajat koherensi kompleks $\gamma_{12}$ diketahui, model lebih umum pada titik sensor:

$$I=I_1+I_2+2\sqrt{I_1 I_2}\operatorname{Re}[\gamma_{12}\exp(i\Delta\phi)],$$

sehingga $V=2\sqrt{I_1I_2}|\gamma_{12}|/(I_1+I_2)$ untuk pola sinusoidal stabil. Ini menunjukkan **kontras bergantung pada kesetimbangan intensitas DAN koherensi**. Untuk $\gamma_{12}=0$, tidak ada suku interferensi rata-rata meskipun masing-masing gelombang tetap ada.

**Hubungan optik–digitisasi:** setelah optika dan sensor, kita memodelkan $I(x,y)=A(x,y)+B(x,y)\cos[\phi(x,y)]+\eta(x,y)$, dengan $A,B$ dalam satuan sinyal sensor (mis. elektron atau nilai linear terkalibrasi), $\phi$ dalam rad, dan $\eta$ adalah noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi': 105, 'axes.grid': False,
                     'image.interpolation': 'nearest'})
print('NumPy:', np.__version__)

In [ ]:
# Membuktikan ekspansi intensitas dari medan kompleks (semua nilai sintetis)
rng = np.random.default_rng(20261116)
ny, nx = 180, 256
y, x = np.mgrid[:ny, :nx]
phi1 = 2*np.pi*(0.085*x + 0.018*y)
phi2 = 0.85*np.exp(-((x-132)**2+(y-90)**2)/(2*40**2))
I1, I2 = 0.36, 0.25
E1 = np.sqrt(I1)*np.exp(1j*phi1)
E2 = np.sqrt(I2)*np.exp(1j*phi2)
I_field = np.abs(E1+E2)**2
I_analytic = I1+I2+2*np.sqrt(I1*I2)*np.cos(phi1-phi2)
assert np.allclose(I_field, I_analytic, atol=1e-12)
V_pred = 2*np.sqrt(I1*I2)/(I1+I2)
print(f'Verifikasi ekspansi kompleks: OK; visibilitas ideal V = {V_pred:.4f}')
fig, ax = plt.subplots(1, 3, figsize=(13,3.7), constrained_layout=True)
for a, data, title in zip(ax,[phi1-phi2,I_field,I_analytic],
                           ['Beda fase [rad]', 'Intensitas dari medan', 'Intensitas dari rumus']):
    im=a.imshow(data,cmap='gray' if 'Intensitas' in title else 'twilight');
    a.set(title=title,xlabel='x [piksel]',ylabel='y [piksel]');plt.colorbar(im,ax=a)
plt.show()

## 2 · Visibilitas vs koherensi


Citra fringe dapat menghilang tanpa mengubah panjang gelombang; cukup dengan menurunkan $|\gamma_{12}|$ atau membuat intensitas kedua lengan menjadi tidak seimbang sehingga kontras memburuk. Jangan mencampur **koherensi fisik** dengan **noise sensor**: keduanya dapat menurunkan kemampuan rekonstruksi dengan mekanisme berbeda.

In [ ]:
gammas=[1.0,0.6,0.15,0.0]
fig,axes=plt.subplots(1,4,figsize=(13,2.8),constrained_layout=True)
for g,a in zip(gammas,axes):
    frame=I1+I2+2*np.sqrt(I1*I2)*g*np.cos(phi1-phi2)
    a.imshow(frame,cmap='gray',vmin=0,vmax=1.25)
    a.set_title(f'|gamma|={g:.2f}');a.axis('off')
plt.show()
assert V_pred<=1+1e-12

## 3 · Akuisisi digital: shot noise, read noise, kuantisasi, saturasi


Asumsikan intensitas linear ternormalisasi $I_n\in[0,1]$ diubah menjadi jumlah fotoelektron rata-rata $\mu=N_{\max}I_n$. Model noise sederhana:

$$N_e\sim\operatorname{Poisson}(\mu),\quad N_\mathrm{read}\sim\mathcal N(0,\sigma_r^2),\quad D=Q_b\!\left(\operatorname{clip}\left[(N_e+N_\mathrm{read})/N_{\max},0,1\right]\right).$$

$N_{\max}$ adalah parameter skala elektron ilustratif (bukan full-well kamera yang telah dikarakterisasi), $\sigma_r$ dalam elektron, dan $Q_b$ kuantisasi linear $b$-bit. Pada sistem nyata perlu diperhitungkan respons nonlinear, PRNU/DSNU, dark current, demosaicing, dan kalibrasi radiometrik.

In [ ]:
def simulate_camera(normalized_irradiance, full_scale_e=2500, read_sigma_e=4.0,
                    bit_depth=12, seed=0):
    # Sensor sintetis dengan shot+read noise, clipping, dan ADC linear.
    rng=np.random.default_rng(seed)
    mu=np.maximum(normalized_irradiance,0)*full_scale_e
    electrons=rng.poisson(mu)+rng.normal(0,read_sigma_e,mu.shape)
    codes=np.rint(np.clip(electrons/full_scale_e,0,1)*(2**bit_depth-1))
    return codes.astype(np.uint16), codes/(2**bit_depth-1)

ideal=(0.5+0.39*np.cos(phi1-phi2))
codes, measured=simulate_camera(ideal,full_scale_e=1200,read_sigma_e=5,seed=21)
fig,axes=plt.subplots(1,3,figsize=(13,3.5),constrained_layout=True)
for a,img,label in zip(axes,[ideal,measured,measured-ideal],
                       ['Model intensitas linear', 'Citra digital sintetis (12-bit)', 'Residual kamera']):
    h=a.imshow(img,cmap='gray' if label!='Residual kamera' else 'coolwarm');
    a.set(title=label,xlabel='x [px]',ylabel='y [px]');plt.colorbar(h,ax=a)
plt.show()
print('RMSE noise+kuantisasi:',np.sqrt(np.mean((measured-ideal)**2)))
assert codes.min()>=0 and codes.max()<=4095

## 4 · Sampling spasial dan aliasing


Jika frekuensi carrier $f_x$ dinyatakan dalam **siklus/piksel**, frekuensi Nyquist kamera adalah $|f_x|<0.5$ siklus/piksel. Sekalipun carrier memenuhi Nyquist, sideband yang membawa bandwidth objek harus tetap dipisahkan dari DC dan dari sideband konjugat; karena itu Fourier phase retrieval membutuhkan margin frekuensi tambahan. Jika $f_x=0.62$ siklus/piksel, sampling mereduksinya menjadi alias $f_a=|f_x-1|=0.38$ siklus/piksel.

In [ ]:
n_samples=256
xs=np.arange(n_samples)
fig,axs=plt.subplots(2,2,figsize=(11,5),constrained_layout=True)
for row,f in enumerate([0.12,0.62]):
    sig=np.cos(2*np.pi*f*xs)
    axs[row,0].plot(xs[:70],sig[:70],'-o',markersize=2)
    axs[row,0].set(title=f'Sampling {f:.2f} siklus/piksel',xlabel='x [px]',ylabel='sinyal')
    spectrum=np.abs(np.fft.fftshift(np.fft.fft(sig)))
    freq=np.fft.fftshift(np.fft.fftfreq(n_samples))
    axs[row,1].plot(freq,spectrum)
    axs[row,1].set(xlabel='Frekuensi spasial [siklus/px]',ylabel='|FFT|')
plt.show()
assert np.allclose(np.cos(2*np.pi*0.62*xs),np.cos(2*np.pi*(-0.38)*xs),atol=1e-12)

## 5 · Eksperimen dan validasi mandiri


**Eksperimen A (koherensi):** variasikan $I_1/I_2$ dan $|\gamma_{12}|$; catat selisih visibilitas model dan pengukuran dari citra noiseless. **Eksperimen B (kamera):** variasikan electron budget, bit depth, dan persentase piksel saturasi. **Eksperimen C (sampling):** uji carrier 0.05 hingga 0.65 siklus/piksel dan tentukan alias yang teramati.

**Riset mini (HOTS):** rancang kriteria kualitas citra agar metode Fourier satu-frame dapat bekerja, sekaligus jelaskan mengapa memenuhi Nyquist saja belum cukup. Tuliskan asumsi, plot pembanding, dan nilai kuantitatif.

**Checkpoint:** $I$ tidak boleh negatif; $0\le V\le1$ pada definisi intensitas ideal; fase tidak boleh diberi label 'meter'; semua data adalah simulasi.

## Referensi ilmiah


- Goodman, J. W., *Statistical Optics*, edisi kedua, Wiley (2015) — teori koherensi dan speckle (rujukan bibliografis umum; akses teks penuh tidak diasumsikan).
- Takeda, M., Ina, H., & Kobayashi, S. (1982), *Fourier-transform method of fringe-pattern analysis for computer-based topography and interferometry*, JOSA **72**, 156–160. https://doi.org/10.1364/JOSA.72.000156
- NIST, *Guidelines for Evaluating and Expressing the Uncertainty of NIST Measurement Results*: https://www.nist.gov/pml/nist-technical-note-1297

<!-- AUDIT-RESEARCH-2026 -->

## 6 · Verifikasi statistik kamera: shot noise, read noise, dan saturasi

**Hipotesis yang diuji:** sebelum clipping dan kuantisasi, varians jumlah elektron sensor ideal bertambah secara linear terhadap rata-rata elektron, dengan intersep $\sigma_r^2$:

$$\mathbb E[Y_e]=\mu_e,\qquad \operatorname{Var}(Y_e)=\mu_e+\sigma_r^2,\qquad Y_e\equiv \operatorname{Poisson}(\mu_e)+\mathcal N(0,\sigma_r^2).$$

Hubungan tersebut berlaku untuk pembacaan **elektron linear sebelum ADC**. Setelah clipping, demosaicing, nonlinearitas, atau korelasi spasial, varians tidak lagi harus berbentuk ini. Pengujian berikut menghitung statistik ensemble pada level iluminasi berbeda; satu citra kamera tidak cukup untuk menyimpulkan karakteristik shot noise dengan memisahkan noise spasial dari tekstur objek.

**Preregistrasi simulasi:** 18 level iluminasi, 16000 replikasi/level, seed tetap, `read_sigma_e=5`, dan model ideal tanpa clipping. Kriteria kesesuaian dipilih sebelum simulasi: galat kemiringan $<5\%$ dan galat intersep $<20\%$. Tampilkan kurva variance–mean dan estimasi ketidakpastian standar kemiringan dari regresi linear; pengujian bukan karakterisasi sensor fisik.

In [ ]:
read_sigma_e = 5.0
levels_e = np.linspace(80., 2400., 18)
rng_cam = np.random.default_rng(1601)
samples_e = rng_cam.poisson(levels_e[:, None], size=(len(levels_e), 16000)) + rng_cam.normal(0, read_sigma_e, (len(levels_e), 16000))
means_e = samples_e.mean(axis=1)
variances_e = samples_e.var(axis=1, ddof=1)
fit_cam = stats.linregress(means_e, variances_e)
print(f'Var-mean slope={fit_cam.slope:.3f} ± {fit_cam.stderr:.3f}; intercept={fit_cam.intercept:.1f} e² (truth={read_sigma_e**2:.1f})')
assert abs(fit_cam.slope-1.) < 0.05
assert abs(fit_cam.intercept-read_sigma_e**2)/read_sigma_e**2 < 0.20
fig, ax = plt.subplots(figsize=(7.8, 4))
ax.scatter(means_e, variances_e, label='Ensemble synthetic')
ax.plot(means_e, fit_cam.intercept+fit_cam.slope*means_e, label='Linear fit')
ax.plot(levels_e, levels_e+read_sigma_e**2, '--',label='Analytic expectation')
ax.set(xlabel='Rata-rata sinyal [elektron]',ylabel='Varians temporal [elektron²]',title='Kalibrasi model noise sintetis')
ax.legend(); plt.show()
# Model saturasi: clipping mengubah kemiringan dan bias, jangan diperlakukan sebagai noise Gaussian
mu_sat = np.linspace(500, 4000, 8)
sat = rng_cam.poisson(mu_sat[:, None], size=(len(mu_sat),2000)).clip(0,2500)
print('Bias pembacaan pada level tertinggi setelah saturasi:',round(float(sat[-1].mean()-mu_sat[-1]),1),'elektron')
assert sat[-1].mean() < mu_sat[-1]-1000